In [ ]:
import pandas as pd

from geofeatureviz.data import OverpassAPIHandler, anki_connector, path_settings

# OSM IDs of German Mountain Ranges
In this notebook, I show how I obtained the OpenStreetMap IDs of the German mountain ranges that I want to have in my Anki deck.

I already had an Anki deck, and I use the `anki_connector` to get the mountains in the deck. For some, the name in the Anki deck couldn't be found in OSM, so I had to change it to the name that can be found in OSM.

In [ ]:
deck_name = path_settings.anki_mountain_deck
anki_mountain_df = anki_connector.deck_to_df(deck_name)
anki_mountain_df = anki_mountain_df[
    anki_mountain_df["NoteType"] == path_settings.anki_mountain_note_type
]
anki_mountain_df["osm_name"] = anki_mountain_df["Gebirge"].replace(
    {
        "Alpen (nur Deutschland)": "Alps",
        "Oberpfälzer Wald": "Naturpark Oberpfälzer Wald",
        "Rothaargebirge": "Naturpark Sauerland-Rothaargebirge",
    }
)
osm_name_to_name = dict(zip(anki_mountain_df["osm_name"], anki_mountain_df["Gebirge"]))
print(f"There are {len(anki_mountain_df)} German mountain ranges.")

Afterward, I search for all relations that have these names using the Overpass API.

In [ ]:
regex = [f"^{f}$" for f in anki_mountain_df["osm_name"]]
regex = "|".join(regex)
overpass_query = f'relation["name"~"{regex}"]'

api_handler = OverpassAPIHandler(
    file_path=path_settings.data_raw_dir
    / "osm_mountains"
    / "german_mountains_body_name.json",
)
api_handler.create_query(overpass_query)

api_handler.get()
mountain_df = api_handler.parse_json()
mountain_df = mountain_df.rename(columns={"name": "osm_name"})

print(f"The response currently has {len(mountain_df)} elements.")

The response has too many entries, since there are duplicates and unwanted things. First, I try to remove duplicates: I remove all duplicates, where one of the duplicates has the OSM tag `natural=mountain_range`. Afterward, I manually checked the elements and gathered the IDs that I didn't want and removed them from the list.

In [ ]:
# remove duplicates if one of the duplicates has natural=mountain_range
duplicate_mask = mountain_df.duplicated("osm_name", keep=False)
mr_mask = mountain_df["natural"].isin(["mountain_range"])
keep_mr_mask = mountain_df.groupby("osm_name")["natural"].transform(
    lambda s: (s == "mountain_range").any()
)
mask = ~duplicate_mask | mr_mask | ~keep_mr_mask
mountain_df = mountain_df[mask]

# remove unwanted ids
unwanted_ids = [
    3133559,
    2750949,
    20773327,
    18455408,
    2142664,
    3856617,
    2142626,
    18543432,
    2122777,
    15775245,
]
mountain_df = mountain_df[~mountain_df.id.isin(unwanted_ids)]

print(f"Now the response has {len(mountain_df)} elements.")
missing = set(anki_mountain_df["osm_name"]) - set(mountain_df["osm_name"])
print(f"The following mountain ranges are missing: {missing}")

I had to find out, that some mountain ranges are not defined as *relation*, but as *way*. Here, it was most convenient to just manually search for them three on [OSM](openstreetmap.org) and add them manually. Before, I only keep the necessary columns and add a column for the OSM type to track whether an element is a *relation* or *way*.

In [ ]:
mountain_df["name"] = mountain_df["osm_name"].replace(osm_name_to_name)
mountain_df = mountain_df.rename(columns={"id": "osm_id"})
mountain_df["osm_type"] = "relation"
mountain_df = mountain_df[["name", "osm_id", "osm_name", "osm_type"]]

new_rows = pd.DataFrame(
    {
        "name": ["Sächsische Schweiz", "Erzgebirge", "Hunsrück", "Rhön"],
        "osm_id": [202443898, 360359268, 544102426, 571378856],
        "osm_name": ["Sächsische Schweiz", "Erzgebirge", "Hunsrück", "Rhön"],
        "osm_type": "way",
    }
)
mountain_df = pd.concat([mountain_df, new_rows], ignore_index=True)
mountain_df = mountain_df.sort_values(by="osm_name").reset_index(drop=True)

# Check correctness
missing = set(anki_mountain_df["Gebirge"]) - set(mountain_df["name"])
print(f"Missing: {missing}")
has_duplicates = mountain_df["osm_name"].duplicated().any()
print(f"Has duplicates: {has_duplicates}")

mountain_df.to_csv(str(path_settings.german_mountains_osm_id_path))